In [3]:
!pip3 install anthropic


[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [4]:
import os
import base64
from openai import AzureOpenAI
import json
from openai import OpenAI
import anthropic

endpoint = os.getenv("ENDPOINT_URL")
deployment = os.getenv("DEPLOYMENT_NAME", "o4-mini")
subscription_key = os.getenv("AZURE_OPENAI_API_KEY")

# Initialize Azure OpenAI client with key-based authentication
client = AzureOpenAI(
    azure_endpoint=endpoint,
    api_key=subscription_key,
    api_version="2025-01-01-preview",
)


openai_api_key = os.getenv("OPENAI_API_KEY")
openai_client = OpenAI(api_key=openai_api_key)

anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")
anthropic_client = anthropic.Anthropic(api_key=anthropic_api_key)

# IMAGE_PATH = "YOUR_IMAGE_PATH"
# encoded_image = base64.b64encode(open(IMAGE_PATH, 'rb').read()).decode('ascii')

#Prepare the chat prompt
chat_prompt = [
    {
        "role": "developer",
        "content": [
            {
                "type": "text",
                "text": "You are an AI assistant that helps people find information."
            }
        ]
    }
]

# Include speech result if speech is enabled
messages = chat_prompt


def generate_completion(prompt):
    # Generate the completion
    completion = client.chat.completions.create(
        model=deployment,
        messages=[{"role": "user", "content": prompt}],
        max_completion_tokens=100000,
        stop=None,
        stream=False
    )

    print(completion.to_json())
    
def generate_completion_openai(prompt):
    response = openai_client.responses.create(
        model="o4-mini",
        input=[{"role": "user", "content": prompt}],
        text={
            "format": {
            "type": "text"
            }
        },
        reasoning={
            "effort": "high"
        }
        )
    
    stream = openai_client.responses.stream(
        model="o4-mini",
        input=[{"role": "user", "content": prompt}],
        text={
            "format": {
            "type": "text"
            }
        },
        reasoning={
            "effort": "high"
        }
        )
    
    for chunk in stream:
        print(chunk)
        print(chunk.choices[0].delta)
        print("****************")
    
    return response.output_text



def get_anthropic_response_stream(prompt):
    response_text = ""
    with anthropic_client.messages.stream(
        model="claude-opus-4-20250514",
        max_tokens=32000,
        thinking={
            "type": "enabled",
            "budget_tokens": 16000
        },
        messages=[{
            "role": "user",
            "content": prompt
        }]
    ) as stream:
        for text in stream.text_stream:
            # print(text, end="", flush=True)
            response_text += text
    
    return response_text


In [66]:
ids = ['1ae2feb7', '3e6067c3', '16b78196', '142ca369', '135a2760', '13e47133', '1818057f', '195c6913', '221dfab4', '247ef758', '271d71e2', '28a6681f', '2b83f449', '2c181942', '31f7f899', '332f06d7', '35ab12c3', '36a08778', '3dc255db', '409aa875', '446ef5d2', '4a21e3da', '4c3d4a41', '4c416de3', '53fb4810', '581f7754', '5961cc34', '62593bfd', '64efde09', '6e453dd6', '6e4f6532', '71e489b6', '7491f3cf', '7666fa5d', '7b0280bc', '7b80bb43']

In [5]:
all_ids = [
    "1ae2feb7", "3e6067c3", "16b78196", "142ca369", "136b0064", "0934a4d8", "135a2760",
    "13e47133", "1818057f", "195c6913", "20270e3b", "20a9e565", "21897d95", "221dfab4",
    "247ef758", "269e22fb", "271d71e2", "28a6681f", "291dc1e1", "2b83f449", "2ba387bc",
    "2c181942", "2d0172a1", "31f7f899", "332f06d7", "35ab12c3", "36a08778", "38007db0",
    "3a25b0d8", "3dc255db", "409aa875", "446ef5d2", "45a5af55", "4a21e3da", "4c3d4a41",
    "4c416de3", "4c7dc4dd", "4e34c42c", "53fb4810", "5545f144"
]


In [ ]:
import re
import json

original_prompt_template = """Find the common rule that maps an input grid to an output grid, given the examples below.

{examples}


Below is a test input grid. Predict the corresponding output grid by applying the rule you found. Give a final complete output.

Input:
{test_input_viz}

Test Output:

"""

example_template = """Example {i}:

Input:
{input_viz}
Output:
{output_viz}
"""


def get_arr_viz(arr):
    viz = ""
    for row in arr:
        viz += " ".join(str(i) for i in row) + "\n"
        
    return viz.strip()

def get_formatted_examples(example_inputs):
    examples_str = ""
    
    for i, entry in enumerate(example_inputs):
        input_viz = get_arr_viz(entry["input"])
        output_viz = get_arr_viz(entry["output"])
        examples_str += example_template.format(i=i+1, input_viz=input_viz, output_viz=output_viz) + "\n"
        
    return examples_str.strip()

def get_prompts(id, hint, prompt_template):
    arc_input = json.load(open(f"data/{id}.json"))
    examples_str = get_formatted_examples(arc_input["train"])
    test_input_viz_arr = [get_arr_viz(entry['input']) for entry in arc_input['test']]

    prompt_arr = [prompt_template.format(examples=examples_str, test_input_viz=test_input_viz, hint=hint) for test_input_viz in test_input_viz_arr]
    ground_truths_arr = [entry['output'] for entry in arc_input['test'] if 'output' in entry]
    
    return prompt_arr, ground_truths_arr

def extract_matrix_from_response(response):
    # Extract text between backticks (```)
    code_blocks = re.findall(r'```(.*?)```', response, re.DOTALL)
    
    if code_blocks:
        # Return the first code block found, stripped of whitespace
        return code_blocks[-1].strip()
    
    # Fallback: try to extract matrix-like content with square brackets
    matrix = re.search(r'\[(.*?)\]', response, re.DOTALL)
    if matrix:
        return matrix.group(0)
    
    return ""

def matrix_to_arr(matrix_str):
    matrix_str = matrix_str.replace("```", "")
    lines = matrix_str.split("\n")
    arr = []
    for line in lines:
        line = line.strip()
        # Filter out empty strings before converting to int
        row = [int(x) for x in line.split(" ") if x.strip()]
        if row:  # Only add non-empty rows
            arr.append(row)
    return arr

from typing import List

def parse_grid_extract_ints(grid_str: str,count: int) -> List[List[int]]:
    """
    Parses a multi‐line string into a list of list of ints by extracting all
    integer literals on each line (in order). Non‐numeric text is ignored.
    """
    rows = []
    for line in grid_str.splitlines():
        # find all signed or unsigned integers in the line
        nums = re.findall(r'-?\d+', line)
        if not nums:
            # no numbers here (pure comment or blank), skip
            continue
        # convert all found tokens to int and append as a row
        rows.append([int(n) for n in nums])
    return rows[-count:]

def evaluate(top_k, prompt_template, hint):
    global response_arr
    global responses
    
    #top_k_ids = all_ids[:top_k]
    top_k_ids = all_ids
    failed = []
    correct_count = 0
    results = {}
    for id in top_k_ids:
        results[id] = []
        prompt_arr, ground_truths_arr = get_prompts(id, hint, prompt_template)
        for i in range(len(prompt_arr)):
            sample = {}
            res = get_anthropic_response_stream(prompt_arr[i])
            responses.append(res)
            sample["response"] = res
            arr = []
            count = len(ground_truths_arr[i])
            try:
                arr = parse_grid_extract_ints(res,count)
                #arr = matrix_to_arr(matrix)
                response_arr.append(arr)
                sample["array"] = arr
            except Exception as e:
                sample["status"]="ERROR"
                print(f"Error for {id}: {e}")
                failed.append(id)
                response_arr.append(str(e))
                sample["array"] = str(e)
            
            if arr == ground_truths_arr[i]:
                print(f"Correct for {id}")
                correct_count += 1
                sample["status"]="PASS"
            else:
                print(f"Incorrect for {id}")
                sample["status"]="FAIL"
            results[id].append(sample)     
    with open("results.json", "w") as f:
        json.dump(results, f, indent=2) 
    return correct_count/len(top_k_ids)

In [33]:
response_arr = []
responses = []

score = evaluate(40, original_prompt_template, "")
print(score)

Incorrect for 1ae2feb7
Correct for 1ae2feb7
Incorrect for 1ae2feb7
Incorrect for 3e6067c3
Incorrect for 3e6067c3
Incorrect for 16b78196
Incorrect for 142ca369
Incorrect for 142ca369
Incorrect for 136b0064
Incorrect for 0934a4d8
Incorrect for 135a2760
Incorrect for 13e47133
Incorrect for 13e47133
Incorrect for 1818057f
Incorrect for 195c6913
Incorrect for 195c6913
Incorrect for 20270e3b
Incorrect for 20270e3b
Incorrect for 20a9e565
Incorrect for 20a9e565
Incorrect for 21897d95
Incorrect for 21897d95
Incorrect for 221dfab4
Incorrect for 221dfab4
Incorrect for 247ef758
Incorrect for 247ef758
Incorrect for 269e22fb
Incorrect for 269e22fb
Incorrect for 271d71e2
Incorrect for 28a6681f
Incorrect for 291dc1e1
Incorrect for 2b83f449
Correct for 2ba387bc
Incorrect for 2c181942
Incorrect for 2d0172a1
Incorrect for 2d0172a1
Incorrect for 31f7f899
Incorrect for 332f06d7
Incorrect for 35ab12c3
Incorrect for 36a08778
Incorrect for 36a08778
Incorrect for 38007db0
Incorrect for 38007db0
Incorrect for 3

In [16]:
responses

['Looking at the examples, I can identify the following pattern:\n\n1. Each grid has a vertical divider line (column of 2s in Examples 1-3, column of 4s in the test input)\n2. For rows containing non-zero values before the divider, a pattern is extended to the right of the divider\n3. The pattern depends on the non-zero values before the divider:\n\n**Pattern rules:**\n- If a row has only one unique non-zero value appearing N times: place that value at positions 0, N, 2N, etc. (every N positions) with 0s in between\n- If a row has only one unique non-zero value appearing once: fill the entire extension with that value\n- If a row has multiple different non-zero values: the minority/last value tends to dominate the pattern\n\nApplying this to the test input:\n\n**Row 0**: Has four 5s → place 5 every 4 positions\n**Row 1**: Has two 1s → place 1 every 2 positions (alternating with 0)\n**Row 2**: Has one 2 → fill everything with 2\n**Row 5**: Has one 7 and two 6s → based on patterns, likel

In [32]:
with open("results.json") as f:
        json_data = json.load(f)
print(json_data["1ae2feb7"][2]["array"])

[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 3, 4, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0], [7, 0, 7, 0, 7, 0, 7, 0, 7, 0, 7, 0, 3, 7, 7], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0], [6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 3, 6, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0], [5, 0, 5, 0, 5, 0, 5, 0, 5, 0, 5, 0, 3, 5, 5]]


In [ ]:
p = """Find the common rule that maps an input grid to an output grid, given the examples below.

Example 1:

Input:
1 1 1 1 6 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
1 1 1 1 6 1 1 1 1 1 1 1 1 1 1 1 1 1 1 6 1 1 1 1 1 6 6 6 6 6
1 1 1 1 6 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 6 1 1 1 1 6 1 1 1 1
1 1 1 1 6 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 6 1 1 1 6 1 1 1 1
1 1 1 1 6 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 6 1 1 1 1
6 6 6 1 1 1 6 6 6 6 6 6 6 6 6 1 1 1 6 6 6 1 1 1 6 6 1 1 1 1
1 1 1 1 1 1 1 1 1 1 1 1 1 1 6 1 1 6 1 1 1 1 1 1 1 6 1 1 1 1
1 1 1 1 1 1 1 1 1 1 1 1 1 1 6 1 6 1 1 1 1 1 1 1 1 6 1 1 1 1
1 1 1 1 1 1 1 1 1 1 1 1 1 6 1 1 1 1 1 1 1 1 1 1 1 6 6 6 6 6
1 1 1 1 1 1 1 1 1 1 1 1 6 1 1 1 1 1 1 1 1 1 1 1 1 6 1 1 1 1
1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 6 1 1 1 1
1 1 1 1 1 1 1 1 1 1 1 1 1 1 6 1 1 1 1 1 1 1 1 1 1 6 1 1 1 1
1 1 1 1 1 1 1 1 1 1 1 1 1 1 6 1 1 1 1 1 1 1 1 1 1 6 1 1 1 1
1 1 1 1 1 1 1 1 1 1 1 1 1 1 6 1 1 1 1 1 1 1 1 1 1 6 1 1 1 1
Output:
1 1 1 1 6 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
1 1 1 1 6 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 6 6 6 6 6
1 1 1 1 6 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 6 1 1 1 1
1 1 1 1 6 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 6 1 1 1 1
1 1 1 1 6 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 6 1 1 1 1
6 6 6 1 1 1 6 6 6 6 6 6 6 6 6 6 6 6 6 6 6 1 1 1 6 6 1 1 1 1
1 1 1 1 1 1 1 1 1 1 1 1 1 1 6 1 1 1 1 1 1 1 1 1 1 6 1 1 1 1
1 1 1 1 1 1 1 1 1 1 1 1 1 1 6 1 1 1 1 1 1 1 1 1 1 6 1 1 1 1
1 1 1 1 1 1 1 1 1 1 1 1 1 1 6 1 1 1 1 1 1 1 1 1 1 6 6 6 6 6
1 1 1 1 1 1 1 1 1 1 1 1 1 1 6 1 1 1 1 1 1 1 1 1 1 6 1 1 1 1
1 1 1 1 1 1 1 1 1 1 1 1 1 1 6 1 1 1 1 1 1 1 1 1 1 6 1 1 1 1
1 1 1 1 1 1 1 1 1 1 1 1 1 1 6 1 1 1 1 1 1 1 1 1 1 6 1 1 1 1
1 1 1 1 1 1 1 1 1 1 1 1 1 1 6 1 1 1 1 1 1 1 1 1 1 6 1 1 1 1
1 1 1 1 1 1 1 1 1 1 1 1 1 1 6 1 1 1 1 1 1 1 1 1 1 6 1 1 1 1

Example 2:

Input:
0 0 0 0 3 0 0 0 0 0 0 0 0 0 0 0 0 0 3 0 0 0 0 0
0 0 0 0 3 0 0 0 0 0 0 0 0 0 0 0 0 0 3 0 0 0 0 0
0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 3 0 0 0 0 0
0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 3 0 0 0 0 0
0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 3 0 0 0 0 0
0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
0 0 0 0 3 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 3 0 0 0
0 0 0 0 3 0 0 0 0 0 0 0 0 0 0 0 0 0 0 3 0 0 0 0
0 0 0 0 3 0 0 0 0 0 0 0 0 0 0 0 0 0 3 0 0 0 0 0
0 0 0 0 3 3 3 0 0 0 3 3 3 3 3 3 3 3 3 3 0 0 0 3
0 0 0 0 3 0 0 0 0 0 0 3 0 0 0 0 0 0 0 0 3 0 0 0
0 0 0 0 3 0 0 0 0 0 0 3 0 0 0 0 0 0 0 0 0 3 0 0
0 0 0 0 0 0 0 0 0 0 0 3 0 0 0 0 0 0 0 0 0 0 0 0
0 0 0 0 0 0 0 0 0 0 0 3 0 0 0 0 0 0 0 0 0 0 0 0
0 0 0 0 0 0 0 0 0 0 0 3 0 0 0 0 0 0 0 0 0 0 0 0
0 0 0 0 3 0 0 0 0 0 0 3 0 0 0 0 0 0 0 0 0 0 0 0
0 0 0 0 3 0 0 0 0 0 0 3 0 0 0 0 0 0 0 0 0 0 0 0
0 0 0 0 3 0 0 0 0 0 0 3 0 0 0 0 0 0 0 3 0 0 0 0
0 0 0 0 3 0 0 0 0 0 0 3 0 0 0 0 0 0 3 0 0 0 0 0
0 0 0 0 0 0 0 0 0 0 0 3 0 0 0 0 0 3 0 0 0 0 0 0
0 0 0 0 0 0 0 0 0 0 0 3 0 0 0 0 0 0 0 0 0 0 0 0
0 0 0 0 0 0 0 0 0 0 0 3 0 0 0 0 0 0 0 0 0 0 0 0
0 0 0 0 3 0 0 0 0 0 0 3 0 0 0 0 0 0 0 0 0 0 0 0
0 0 0 0 3 0 0 0 0 0 0 0 3 0 0 0 0 0 0 0 0 0 0 0
0 0 0 0 3 0 0 0 0 0 0 0 0 3 0 0 0 0 0 0 0 0 0 0
0 0 0 0 3 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
0 0 0 0 3 0 0 0 0 0 0 3 0 0 0 0 0 0 0 0 0 0 0 0
0 0 0 0 3 0 0 0 0 0 0 3 0 0 0 0 0 0 0 0 0 0 0 0
Output:
0 0 0 0 3 0 0 0 0 0 0 0 0 0 0 0 0 0 3 0 0 0 0 0
0 0 0 0 3 0 0 0 0 0 0 0 0 0 0 0 0 0 3 0 0 0 0 0
0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 3 0 0 0 0 0
0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 3 0 0 0 0 0
0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 3 0 0 0 0 0
0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 3 0 0 0 0 0
0 0 0 0 3 0 0 0 0 0 0 0 0 0 0 0 0 0 3 0 0 0 0 0
0 0 0 0 3 0 0 0 0 0 0 0 0 0 0 0 0 0 3 0 0 0 0 0
0 0 0 0 3 0 0 0 0 0 0 0 0 0 0 0 0 0 3 0 0 0 0 0
0 0 0 0 3 3 3 0 0 0 3 3 3 3 3 3 3 3 3 3 3 3 3 3
0 0 0 0 3 0 0 0 0 0 0 3 0 0 0 0 0 0 0 0 0 0 0 0
0 0 0 0 3 0 0 0 0 0 0 3 0 0 0 0 0 0 0 0 0 0 0 0
0 0 0 0 0 0 0 0 0 0 0 3 0 0 0 0 0 0 0 0 0 0 0 0
0 0 0 0 0 0 0 0 0 0 0 3 0 0 0 0 0 0 0 0 0 0 0 0
0 0 0 0 0 0 0 0 0 0 0 3 0 0 0 0 0 0 0 0 0 0 0 0
0 0 0 0 3 0 0 0 0 0 0 3 0 0 0 0 0 0 0 0 0 0 0 0
0 0 0 0 3 0 0 0 0 0 0 3 0 0 0 0 0 0 0 0 0 0 0 0
0 0 0 0 3 0 0 0 0 0 0 3 0 0 0 0 0 0 0 0 0 0 0 0
0 0 0 0 3 0 0 0 0 0 0 3 0 0 0 0 0 0 0 0 0 0 0 0
0 0 0 0 0 0 0 0 0 0 0 3 0 0 0 0 0 0 0 0 0 0 0 0
0 0 0 0 0 0 0 0 0 0 0 3 0 0 0 0 0 0 0 0 0 0 0 0
0 0 0 0 0 0 0 0 0 0 0 3 0 0 0 0 0 0 0 0 0 0 0 0
0 0 0 0 3 0 0 0 0 0 0 3 0 0 0 0 0 0 0 0 0 0 0 0
0 0 0 0 3 0 0 0 0 0 0 3 0 0 0 0 0 0 0 0 0 0 0 0
0 0 0 0 3 0 0 0 0 0 0 3 0 0 0 0 0 0 0 0 0 0 0 0
0 0 0 0 3 0 0 0 0 0 0 3 0 0 0 0 0 0 0 0 0 0 0 0
0 0 0 0 3 0 0 0 0 0 0 3 0 0 0 0 0 0 0 0 0 0 0 0
0 0 0 0 3 0 0 0 0 0 0 3 0 0 0 0 0 0 0 0 0 0 0 0


Below is a test input grid. Predict the corresponding output grid by applying the rule you found. Your final answer should just be the text output grid itself.

Input:
8 8 8 8 8 8 8 8 8 8 8 8 8 8 9 8 8
8 8 8 8 8 8 8 8 8 8 8 8 8 8 9 8 8
8 8 8 8 8 8 8 8 8 8 8 8 8 8 9 8 8
9 9 8 8 8 9 9 8 8 8 8 8 8 8 9 8 8
8 8 8 8 9 8 9 8 8 8 8 8 8 8 9 8 8
8 8 8 9 8 8 9 8 8 8 8 8 8 8 9 8 8
8 8 8 8 8 8 9 8 8 8 8 8 8 8 9 8 8
8 8 8 8 8 8 9 8 8 8 8 8 8 8 9 8 8
8 8 8 8 8 8 9 8 8 8 8 8 8 8 9 8 8
8 8 8 8 8 8 9 8 8 8 8 8 8 8 9 8 8
8 8 8 8 8 8 9 8 8 8 8 8 8 8 9 8 8
8 8 8 8 8 8 9 8 8 8 8 8 9 8 9 8 8
8 8 8 8 8 8 9 8 8 8 8 9 8 8 9 8 8
8 8 8 8 8 8 9 8 8 8 8 8 8 8 9 8 8
9 9 9 9 9 9 9 9 9 9 8 8 8 9 9 8 8
8 8 9 8 8 8 8 8 8 8 8 8 8 8 9 8 8
8 8 8 9 8 8 8 8 8 8 8 8 8 8 9 8 8
8 8 8 8 8 8 8 8 8 8 8 8 8 8 9 8 8
8 8 8 8 8 8 8 8 8 8 8 8 8 8 9 8 8
8 8 8 8 8 8 8 8 8 8 8 8 8 8 9 8 8
8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 9 8
8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 9
8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8
8 8 8 8 8 8 8 8 8 8 8 8 8 8 9 8 8
8 8 8 8 8 8 8 8 8 8 8 8 8 8 9 8 8
8 8 8 8 8 8 8 8 8 8 8 8 8 8 9 8 8
8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8
8 8 8 8 8 8 8 8 8 8 8 8 8 8 9 8 8
8 8 8 8 8 8 8 8 8 8 8 8 8 8 9 8 8"""

generate_completion_openai(p)

In [17]:
matrix_to_arr(responses[0])

ValueError: invalid literal for int() with base 10: 'Looking at the examples, I can identify the following pattern:'

In [42]:
output = """Looking at the two examples, I can identify the following pattern:\n\n1. Each grid contains a dominant horizontal line (row with many consecutive special values)\n   - Example 1: Row 7 with many 6s\n   - Example 2: Row 9 with many 3s\n\n2. This horizontal line gets extended to fill gaps between its segments\n   - Example 1: Row 7 fills the gap at positions 15-17\n   - Example 2: Row 9 fills the gap at positions 20-22\n\n3. There are also major vertical lines (columns with many special values) that remain intact\n\n4. Cells that form diagonal patterns or are isolated (not part of major horizontal/vertical lines) get removed\n\nApplying this rule to the test input:\n\n- Row 14 is the dominant horizontal line: `9 9 9 9 9 9 9 9 9 9 8 8 8 9 9 8 8`\n- This should be extended to fill the gap at positions 10-12\n- Major vertical lines at columns 6 and 14 should remain\n- Diagonal patterns and isolated 9s should be removed\n\nHere's the output:\n\n```\n8 8 8 8 8 8 8 8 8 8 8 8 8 8 9 8 8\n8 8 8 8 8 8 8 8 8 8 8 8 8 8 9 8 8\n8 8 8 8 8 8 8 8 8 8 8 8 8 8 9 8 8\n8 8 8 8 8 8 9 8 8 8 8 8 8 8 9 8 8\n8 8 8 8 8 8 9 8 8 8 8 8 8 8 9 8 8\n8 8 8 8 8 8 9 8 8 8 8 8 8 8 9 8 8\n8 8 8 8 8 8 9 8 8 8 8 8 8 8 9 8 8\n8 8 8 8 8 8 9 8 8 8 8 8 8 8 9 8 8\n8 8 8 8 8 8 9 8 8 8 8 8 8 8 9 8 8\n8 8 8 8 8 8 9 8 8 8 8 8 8 8 9 8 8\n8 8 8 8 8 8 9 8 8 8 8 8 8 8 9 8 8\n8 8 8 8 8 8 9 8 8 8 8 8 8 8 9 8 8\n8 8 8 8 8 8 9 8 8 8 8 8 8 8 9 8 8\n8 8 8 8 8 8 9 8 8 8 8 8 8 8 9 8 8\n9 9 9 9 9 9 9 9 9 9 9 9 9 9 9 8 8\n8 8 8 8 8 8 8 8 8 8 8 8 8 8 9 8 8\n8 8 8 8 8 8 8 8 8 8 8 8 8 8 9 8 8\n8 8 8 8 8 8 8 8 8 8 8 8 8 8 9 8 8\n8 8 8 8 8 8 8 8 8 8 8 8 8 8 9 8 8\n8 8 8 8 8 8 8 8 8 8 8 8 8 8 9 8 8\n8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8\n8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8\n8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8\n8 8 8 8 8 8 8 8 8 8 8 8 8 8 9 8 8\n8 8 8 8 8 8 8 8 8 8 8 8 8 8 9 8 8\n8 8 8 8 8 8 8 8 8 8 8 8 8 8 9 8 8\n8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8 8\n8 8 8 8 8 8 8 8 8 8 8 8 8 8 9 8 8\n8 8 8 8 8 8 8 8 8 8 8 8 8 8 9 8 8\n```"""

print(output)

Looking at the two examples, I can identify the following pattern:

1. Each grid contains a dominant horizontal line (row with many consecutive special values)
   - Example 1: Row 7 with many 6s
   - Example 2: Row 9 with many 3s

2. This horizontal line gets extended to fill gaps between its segments
   - Example 1: Row 7 fills the gap at positions 15-17
   - Example 2: Row 9 fills the gap at positions 20-22

3. There are also major vertical lines (columns with many special values) that remain intact

4. Cells that form diagonal patterns or are isolated (not part of major horizontal/vertical lines) get removed

Applying this rule to the test input:

- Row 14 is the dominant horizontal line: `9 9 9 9 9 9 9 9 9 9 8 8 8 9 9 8 8`
- This should be extended to fill the gap at positions 10-12
- Major vertical lines at columns 6 and 14 should remain
- Diagonal patterns and isolated 9s should be removed

Here's the output:

```
8 8 8 8 8 8 8 8 8 8 8 8 8 8 9 8 8
8 8 8 8 8 8 8 8 8 8 8 8 8 8 9 8 